In [ ]:
import pandas as pd
#!pip install anthropic
import anthropic

client = anthropic.Anthropic(api_key="sk-...")  # replace with your actual API key

# ── paths ────────────────────────────────────────────────────────────────────

LEARN_PATH = "D:\\emory books\\2026 spring\\qtm 329\\winograd\\parsed_entries_learn.csv"   # has: sentence, option1, option2, answer, filled_word
EVAL_PATH  = "D:\\emory books\\2026 spring\\qtm 329\\winograd\\parsed_entries_eval.csv"    # has: sentence, option1, option2
OUTPUT_PATH = "D:\\emory books\\2026 spring\\qtm 329\\winograd\\parsed_entries_eval_preds.csv"
N_EXAMPLES  = 15           # few-shot examples from learning set (balanced across 1 & 2)
# ─────────────────────────────────────────────────────────────────────────────

def build_few_shot(learn_df, n=15):
    sampled = (
        learn_df.groupby("answer", group_keys=False)
        .apply(lambda x: x.sample(min(n // 2, len(x)), random_state=42))
        .reset_index(drop=True)
    )
    lines = []
    for _, row in sampled.iterrows():
        lines.append(
            f"Sentence: {row['sentence']}\n"
            f"Option 1: {row['option1']}\n"
            f"Option 2: {row['option2']}\n"
            f"Answer: {int(row['answer'])}"
        )
    return "\n\n".join(lines)


def predict(sentence, option1, option2, few_shot_context):
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=5,
        system=(
            "You are a pronoun resolution classifier.\n"
            "Given a sentence with a pronoun and two candidate referents, "
            "decide which option the pronoun refers to.\n"
            "Reply with ONLY '1' or '2'. No explanation.\n\n"
            "Examples:\n\n"
            f"{few_shot_context}"
        ),
        messages=[{
            "role": "user",
            "content": (
                f"Sentence: {sentence}\n"
                f"Option 1: {option1}\n"
                f"Option 2: {option2}\n"
                f"Answer:"
            )
        }]
    )
    raw = response.content[0].text.strip()
    return int(raw[0]) if raw[0] in ("1", "2") else -1  # -1 = parse failure


def main():
    learn_df = pd.read_csv(LEARN_PATH)
    eval_df  = pd.read_csv(EVAL_PATH)

    few_shot = build_few_shot(learn_df, N_EXAMPLES)
    print(f"Built few-shot context from {N_EXAMPLES} examples.\n")

    predictions = []
    total = len(eval_df)
    for i, row in eval_df.iterrows():
        pred = predict(row["sentence"], row["option1"], row["option2"], few_shot)
        predictions.append(pred)
        print(f"[{i+1}/{total}] → {pred}")

    eval_df["claude_answer"] = predictions
    eval_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nDone! Predictions saved to: {OUTPUT_PATH}")
    print(f"Parse failures (-1): {predictions.count(-1)}")


if __name__ == "__main__":
    main()

C:\Users\srijo\AppData\Local\Temp\ipykernel_37780\4211197702.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(n // 2, len(x)), random_state=42))


Built few-shot context from 15 examples.



BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CZxnSHVPTSMMDH45oboLx'}